# Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_poisson_deviance, mean_gamma_deviance
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from datasets import Dataset
import torch
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

2026-05-07 14:14:54.963771: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-07 14:14:55.541574: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/dkusmenko/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving

# Initialize API Key

In [2]:
load_dotenv()
# Or use `os.getenv('GEMINI_API_KEY')` to fetch an environment variable.
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

# 3. This line is exactly the same as in your image
# It passes the key (which you just fetched) to the client
if not GEMINI_API_KEY:
    print("Error: 'GEMINI_API_KEY' not found.")
    print("Please make sure it is set in your .env file and you have run load_dotenv().")
else:
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("Client configured successfully.")

Client configured successfully.


In [3]:
for m in client.models.list():
  if 'embedContent' in m.supported_actions:
    print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


# Load in Embedding Model

In [ ]:
MODEL_ID = "gemini-embedding-2"

# Load in Data

In [ ]:
# ==========================================
# DATA LOADING & PREPROCESSING
# ==========================================
# Load in freMTPL2freq dataset
dataset = fetch_openml(data_id=41214, as_frame=True)
full_df = dataset.frame

# Clean basic types first
full_df['ClaimNb'] = pd.to_numeric(full_df['ClaimNb'])
full_df['Exposure'] = pd.to_numeric(full_df['Exposure'])
full_df['Exposure'] = full_df['Exposure'].clip(upper=1.0)
full_df['Frequency'] = full_df['ClaimNb'] / full_df['Exposure']

# mapping for contextualized factors
brand_mapping = {'B1': 'Renault, Nissan, or Citroen', 'B2': 'Renault, Nissan, or Citroen',
                 'B3': 'Volkswagen, Audi, Skoda, or Seat', 'B4': 'Opel, General Motors, or Ford',
                 'B5': 'Opel, General Motors, or Ford','B6': 'Fiat', 'B10':'Mercedes, Chrysler, or BMW',
                 'B11':'Mercedes, Chrysler, or BMW', 'B12': 'Japanese (except Nissan) or Korean', 'B13': 'Other','B14': 'Other' }

region_mapping = {
    "R11": "Île-de-France",
    "R21": "Champagne-Ardenne",
    "R22": "Picardie",
    "R23": "Haute-Normandie",
    "R24": "Centre",
    "R25": "Basse-Normandie",
    "R26": "Bourgogne",
    "R31": "Nord–Pas-de-Calais",
    "R41": "Lorraine",
    "R42": "Alsace",
    "R43": "Franche–Comté",
    "R52": "Pays de la Loire",
    "R53": "Bretagne",
    "R54": "Poitou–Charentes",
    "R72": "Aquitaine",
    "R73": "Midi–Pyrénées",
    "R74": "Limousin",
    "R82": "Rhône–Alpes",
    "R83": "Auvergne",
    "R91": "Languedoc–Roussillon",
    "R93": "Provence–Alpes–Côte d’Azur",
    "R94": "Corse"
}

area_mapping = {
    "A": "rural area",
    "B": "semi-rural area",
    "C": "suburban-fringe area",
    "D": "suburban area",
    "E": "urban area",
    "F": "urban center"
}

gas_mapping = {
    "'Diesel'": "Diesel",
    "'Regular'": "Regular"

}

full_df["VehBrand"] = full_df["VehBrand"].map(brand_mapping)
full_df["Region"] = full_df["Region"].map(region_mapping)
full_df["Area"] = full_df["Area"].map(area_mapping)
full_df["VehGas"] = full_df["VehGas"].map(gas_mapping)


In [ ]:
# Load the split indices
df_splits = pd.read_csv('freMTPL2freq_split_indices.csv')

# Ensure IDpol is the same type in both dataframes for a clean merge
full_df['IDpol'] = full_df['IDpol'].astype(int)
df_splits['IDpol'] = df_splits['IDpol'].astype(int)

# Merge the dataset with the split indicators
# We use a left join to keep the original data rows
df_merged = full_df.merge(df_splits, on='IDpol', how='left')

# Create the subsets based on the indicator columns
train_df = df_merged[df_merged['is_train'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()
test_df = df_merged[df_merged['is_test'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()
finetune_df = df_merged[df_merged['is_finetune'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()

# Print results
print(f"Total rows: {len(full_df)}")
print(f"Train rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Finetune rows: {len(finetune_df)}")

# Inspect the training set
print(train_df.head())

# Generate Prompts

In [ ]:
# ==========================================
# 2. SERIALIZATION (Tabular -> Text)
# ==========================================
def serialize_row(row):
    """
    Converts a row of insurance covariates into a natural language prompt.
    Uses a fixed template for consistency between Training and Inference.
    """
    # Handling categorical values cleanly
    veh_brand = str(row['VehBrand']).strip()
    veh_gas = str(row['VehGas']).strip()
    area = str(row['Area']).strip()
    region = str(row['Region']).strip()
    
    return (f"""You are an auto insurance underwriter. Evaluate the risk level of a policyholder based strictly on the following insurance-related information:
- Policyholder Age: {row['DrivAge']} years old (in France people can drive starting at age 18)
- Land Type: {area}
- Region: {region}, France
- Population density: {row['Density']} people/km2 (average density is 1792 people/km2)
- Vehicle: {veh_brand}
- Vehicle Age: {row['VehAge']} years old
- Fuel type: {veh_gas} (either Diesel or Regular Gasoline)
- Power class: {row['VehPower']} (min = 4, max = 15)
- Bonus-Malus score: {row['BonusMalus']} (scored between 50 and 230 with entrance level 100, <100 means bonus, >100 means malus)"""
    )

# Apply serialization
print("Serializing rows to text...")
train_df['text_desc'] = train_df.apply(serialize_row, axis=1)
test_df['text_desc'] = test_df.apply(serialize_row, axis=1)

Successfully generated 50 prompts.

--- PRINTING FIRST PROMPT AS AN EXAMPLE ---
You are an auto insurance underwriter. Evaluate the risk level of a policyholder based strictly on the following insurance-related information:
- Policyholder Age: 55 years old (in France people can drive starting at age 18)
- Land Type: suburban area 
- Region: Rhône–Alpes, France
- Population density: 1217 people/km2 (average density is 1792 people/km2)
- Vehicle: Japanese (except Nissan) or Korean
- Vehicle Age: 0 years old
- Fuel type: Regular (either Diesel or Regular Gasoline)
- Power class: 5 (min = 4, max = 15)
- Bonus-Malus score: 50 (scored between 50 and 230 with entrance level 100, <100 means bonus, >100 means malus)



## Example Prompt

You are an auto insurance underwriter. Evaluate the risk level of a policyholder based strictly on the following insurance-related information:
- Policyholder Age: 44 years old (in France people can drive starting at age 18)
- Land Type: urban center
- Region: Île-de-France, France
- Population density: 27000 people/km2 (average density is 1792 people/km2)
- Vehicle: Japanese (except Nissan) or Korean
- Vehicle Age: 0 years old
- Fuel type: Regular (either Diesel or Regular Gasoline)
- Power class: 9 (min = 4, max = 15)
- Bonus-Malus score: 76 (scored between 50 and 230 with entrance level 100, <100 means bonus, >100 means malus)

# Function to retrieve embeddings

In [ ]:
import time
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from google.genai import types
from google.genai.types import EmbedContentConfig
import os  


config = EmbedContentConfig(
    task_type="CLASSIFICATION",
)

def make_embed_text_fn(model):
    def embed_fn(texts_batch: list[str]) -> list[list[float]]:
        result = client.models.embed_content(
            model=model,
            contents=texts_batch,
            config=config
        ).embeddings
        return [embedding.values for embedding in result]
    return embed_fn


def create_embeddings(df, text_column_name="Text", output_dir="full_context_embedding_batches", start_batch_num=0):
    """
    Applies embeddings and saves each batch to a parquet file
    in 'output_dir' to allow for resumption.
    """
    # --- 2. Create the output directory if it doesn't exist ---
    os.makedirs(output_dir, exist_ok=True)
    
    embed_fn = make_embed_text_fn(MODEL_ID) 
    batch_size = 5 

    # --- 3. Calculate the starting row index ---
    start_index = start_batch_num * batch_size

    if start_index > 0:
        print(f"Resuming from batch {start_batch_num} (row {start_index})...")
    else:
        print(f"Starting from scratch. Batches will be saved to '{output_dir}'")

    # --- 4. Start the loop from the correct index ---
    for i in tqdm(range(start_index, len(df), batch_size)):
        # Get the original slice of the DataFrame for this batch
        batch_df = df.iloc[i : i + batch_size].copy()
        
        # Get texts to embed
        batch_texts = batch_df[text_column_name].tolist()
        
        # Get embeddings
        embeddings = embed_fn(batch_texts)
        
        # Convert embeddings to a float16 DataFrame
        embed_df = pd.DataFrame(
            embeddings, 
            index=batch_df.index, 
            dtype=np.float16
        ).add_prefix("embed_")
        
        

        # --- 5. Save the completed batch to its own file ---
        current_batch_num = i // batch_size
        output_filename = os.path.join(output_dir, f"batch_{current_batch_num}.parquet")
        
        embed_df.to_parquet(output_filename)

        # Sleep to respect rate limits
        time.sleep(2)

    print(f"Embedding process complete. All batches saved to '{output_dir}'.")
    # This function no longer returns a DataFrame.
    # You will combine the files afterward.


# --- HELPER FUNCTIONS --- 

def find_next_batch_num(output_dir="embedding_batches"):
    """
    Checks the output directory to find the last saved batch
    and returns the *next* batch number to start from.
    """
    if not os.path.exists(output_dir):
        return 0  # Directory doesn't exist, start from 0
    
    # List all parquet files
    saved_batches = [f for f in os.listdir(output_dir) if f.endswith(".parquet")]
    
    if not saved_batches:
        return 0  # No batches saved yet, start from 0

    # Find the highest batch number
    last_batch_num = -1
    for f in saved_batches:
        try:
            # Extract number (e.g., from "batch_123.parquet")
            num_str = f.replace("batch_", "").replace(".parquet", "")
            batch_num = int(num_str)
            if batch_num > last_batch_num:
                last_batch_num = batch_num
        except:
            continue 
            
    return last_batch_num + 1


def load_all_batches(output_dir="embedding_batches"):
    """
    Loads all saved parquet files from the directory and
    concatenates them into a single DataFrame.
    """
    saved_batches = [f for f in os.listdir(output_dir) if f.endswith(".parquet")]
    
    # Sort them by batch number to ensure correct order
    saved_batches.sort(key=lambda f: int(f.replace("batch_", "").replace(".parquet", "")))
    
    all_dfs = []
    print(f"Loading {len(saved_batches)} batch files from '{output_dir}'...")
    for f in tqdm(saved_batches):
        filepath = os.path.join(output_dir, f)
        all_dfs.append(pd.read_parquet(filepath))
        
    if not all_dfs:
        print("No batch files found.")
        return pd.DataFrame()

    # Combine all DataFrames into one
    full_df = pd.concat(all_dfs)
    print("All batches loaded and combined.")
    return full_df

# Generate Embeddings and append to dataframe

This will create a directory "full_context_embedding_batches" where the batches are stored. If the API crashes rerun the below code to find the next batch and then run the create embeddings again. Only run the final_df part after all batches have successfully been embedded.

In [ ]:
# Check which batch to start from
train_output_directory = "gemini_train_embeddings"
next_batch = find_next_batch_num(train_output_directory)

In [ ]:
create_embeddings(train_df, text_column_name="text_desc", output_dir=train_output_directory, start_batch_num=next_batch)

Starting from scratch. Batches will be saved to 'full_context_embedding_batches'


  0%|          | 0/10 [00:00<?, ?it/s]

Embedding process complete. All batches saved to 'full_context_embedding_batches'.


In [ ]:
# 1. Load all the embedding batches into one DataFrame
# (Ensure final_df index matches the original test_df/train_df index)
final_train_embeddings = load_all_batches(train_output_directory)

# 2. Select the metadata you want to keep from your original dataframe
meta_columns = ['IDpol', 'ClaimNb', 'Exposure']
metadata = train_df[meta_columns]

# 3. Concatenate them along the columns (axis=1)
# Because final_embeddings preserved the index, they will align perfectly
final_train_combined_df = pd.concat([metadata, final_train_embeddings], axis=1)

# Inspect the result
print(f"Final shape: {final_train_combined_df.shape}")
print(final_train_combined_df[['IDpol', 'ClaimNb', 'Exposure', 'embed_0']].head())

In [ ]:
import numpy as np

# Prepare components for NPZ storage
# X = the embedding matrix 
X_values = final_train_combined_df.filter(like='embed_').values

# y = ClaimNb (The target)
y_values = final_train_combined_df['ClaimNb'].values

# w = Exposure (The weight/offset)
w_values = final_train_combined_df['Exposure'].values

# Save as NPZ for the analysis notebook
np.savez("embeddings/gemini_train_embeddings.npz", X=X_values, y=y_values, w=w_values)

In [ ]:
# Check which batch to start from
test_output_directory = "gemini_test_embeddings"
next_batch = find_next_batch_num(test_output_directory)

In [ ]:
create_embeddings(test_df, text_column_name="text_desc", output_dir=test_output_directory, start_batch_num=next_batch)

In [ ]:
# 1. Load all the embedding batches into one DataFrame
# (Ensure final_df index matches the original test_df/train_df index)
final_test_embeddings = load_all_batches(test_output_directory)

# 2. Select the metadata you want to keep from your original dataframe
meta_columns = ['IDpol', 'ClaimNb', 'Exposure']
metadata = test_df[meta_columns]

# 3. Concatenate them along the columns (axis=1)
# Because final_embeddings preserved the index, they will align perfectly
final_test_combined_df = pd.concat([metadata, final_test_embeddings], axis=1)

# Inspect the result
print(f"Final shape: {final_test_combined_df.shape}")
print(final_test_combined_df[['IDpol', 'ClaimNb', 'Exposure', 'embed_0']].head())

In [ ]:
import numpy as np

# Prepare components for NPZ storage
# X = the embedding matrix
X_values = final_test_combined_df.filter(like='embed_').values

# y = ClaimNb (The target)
y_values = final_test_combined_df['ClaimNb'].values

# w = Exposure (The weight/offset)
w_values = final_test_combined_df['Exposure'].values

# Save as NPZ for the analysis notebook
np.savez("embeddings/gemini_test_embeddings.npz", X=X_values, y=y_values, w=w_values)

